# 12 — Multimodal Fusion Model

Fuse audio and video representations with a compact classifier and compare against unimodal baselines.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

## Fusion strategy
A simple early feature-fusion baseline encodes video frames and audio spectrograms independently, concatenates their embeddings, then predicts the target label. This makes modality contributions easy to ablate.

In [ ]:
import torch
import torch.nn as nn

class FusionClassifier(nn.Module):
    def __init__(self, video_dim=128, audio_dim=64, hidden_dim=128, num_classes=1):
        super().__init__()
        self.video_projection = nn.Linear(video_dim, hidden_dim)
        self.audio_projection = nn.Linear(audio_dim, hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, video_embedding, audio_embedding):
        v = torch.relu(self.video_projection(video_embedding))
        a = torch.relu(self.audio_projection(audio_embedding))
        return self.classifier(torch.cat([v, a], dim=1))

model = FusionClassifier()
print("Fusion head parameters:", sum(p.numel() for p in model.parameters()))

## Required comparisons
1. Video-only baseline.
2. Audio-only baseline.
3. Audio + video fusion.

Use the same split protocol and report accuracy, macro-F1, per-class precision/recall, confusion matrix, and run settings. Verify modality alignment before training.